Use Bayesian Optimization to further optimize hyperparameters by implementing Gaussian Process Regression for intelligent hyperparameter selection without using existing Bayesian optimization libraries.

In [1]:
import pandas as pd
import numpy as np
from itertools import product
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel
from scipy.stats import norm

df = pd.read_csv("heart_disease_train.csv")

features = ["age", "trestbps", "chol", "thalach", "oldpeak"]
target = "target"

df = df[features + [target]].copy()

for col in features + [target]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna()

X = df[features].to_numpy(dtype=float)
y = df[target].to_numpy(dtype=int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 63
Testing samples: 16


In [2]:
def make_model(name, params=None):
    params = params or {}

    if name == "Logistic Regression":
        return make_pipeline(
            StandardScaler(),
            LogisticRegression(
                max_iter=1000, class_weight="balanced", **params
            )
        )

    if name == "Decision Tree":
        return DecisionTreeClassifier(
            class_weight="balanced", random_state=42, **params
        )

    if name == "Random Forest":
        return RandomForestClassifier(
            n_estimators=100, class_weight="balanced",
            random_state=42, **params
        )

    if name == "SVM":
        return make_pipeline(
            StandardScaler(),
            SVC(
                class_weight="balanced", probability=True,
                random_state=42, **params
            )
        )

    if name == "KNN":
        return make_pipeline(
            StandardScaler(),
            KNeighborsClassifier(**params)
        )

param_grids = {
    "Logistic Regression": {
        "C": [0.1, 1, 10],
        "solver": ["liblinear", "lbfgs"]
    },
    "Decision Tree": {
        "max_depth": [3, 5, 10, None],
        "min_samples_leaf": [1, 3, 5]
    },
    "Random Forest": {
        "max_depth": [5, 10, None],
        "min_samples_leaf": [1, 3, 5],
        "max_features": ["sqrt", None]
    },
    "SVM": {
        "C": [0.1, 1, 10],
        "kernel": ["linear", "rbf"]
    },
    "KNN": {
        "n_neighbors": [3, 5, 7, 9],
        "weights": ["uniform", "distance"]
    }
}

In [3]:
def calculate_f1(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))

    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)

    return 2 * precision * recall / max(precision + recall, 1e-12)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def cv_f1(name, params, X_data, y_data):
    scores = []

    for train_idx, val_idx in cv.split(X_data, y_data):
        model = make_model(name, params)
        model.fit(X_data[train_idx], y_data[train_idx])

        pred = model.predict(X_data[val_idx])
        scores.append(calculate_f1(y_data[val_idx], pred))

    return np.mean(scores)

baseline_results = {}

for name in param_grids:
    score = cv_f1(name, {}, X_train, y_train)
    baseline_results[name] = score
    print(name, "Mean CV F1:", round(score, 4))

best_model_name = max(baseline_results, key=baseline_results.get)
print("\nSelected model:", best_model_name)

Logistic Regression Mean CV F1: 0.8452
Decision Tree Mean CV F1: 0.8899
Random Forest Mean CV F1: 0.8889
SVM Mean CV F1: 0.8946


c:\Users\Lenovo\AppData\Local\Python\pythoncore-3.13-64\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\Lenovo\AppData\Local\Python\pythoncore-3.13-64\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\Lenovo\AppData\Local\Python\pythoncore-3.13-64\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\Lenovo\AppData\Local\Python\pythoncore-3.13-64\Lib\site-packages\sklearn\

KNN Mean CV F1: 0.8908

Selected model: SVM


In [4]:
param_grid = param_grids[best_model_name]
param_names = list(param_grid.keys())

candidates = [
    dict(zip(param_names, values))
    for values in product(*param_grid.values())
]

def encode_params(params):
    encoded = []

    for name in param_names:
        choices = param_grid[name]
        index = choices.index(params[name])

        if len(choices) == 1:
            encoded.append(0.0)
        else:
            encoded.append(index / (len(choices) - 1))

    return encoded

candidate_X = np.array([encode_params(p) for p in candidates])

print("Total possible combinations:", len(candidates))
print("Parameters being tuned:", param_names)

Total possible combinations: 6
Parameters being tuned: ['C', 'kernel']


In [5]:
rng = np.random.default_rng(42)

initial_count = min(5, len(candidates))
max_trials = min(15, len(candidates))

tested_indices = list(
    rng.choice(len(candidates), size=initial_count, replace=False)
)

observed_scores = []

# Evaluate initial combinations
for idx in tested_indices:
    score = cv_f1(
        best_model_name, candidates[idx], X_train, y_train
    )
    observed_scores.append(score)

# Gaussian Process surrogate model
kernel = (
    ConstantKernel(1.0) * Matern(length_scale=1.0, nu=2.5)
    + WhiteKernel(noise_level=1e-4)
)

for trial in range(initial_count, max_trials):
    X_observed = candidate_X[tested_indices]
    y_observed = np.array(observed_scores)

    gp = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        n_restarts_optimizer=2,
        random_state=42
    )

    gp.fit(X_observed, y_observed)

    remaining = [
        i for i in range(len(candidates))
        if i not in tested_indices
    ]

    if not remaining:
        break

    X_remaining = candidate_X[remaining]
    mean, std = gp.predict(X_remaining, return_std=True)

    best_so_far = max(observed_scores)
    improvement = mean - best_so_far - 0.01

    z = np.zeros_like(improvement)
    valid = std > 1e-12
    z[valid] = improvement[valid] / std[valid]

    expected_improvement = np.zeros_like(mean)
    expected_improvement[valid] = (
        improvement[valid] * norm.cdf(z[valid])
        + std[valid] * norm.pdf(z[valid])
    )

    next_idx = remaining[int(np.argmax(expected_improvement))]

    next_score = cv_f1(
        best_model_name, candidates[next_idx], X_train, y_train
    )

    tested_indices.append(next_idx)
    observed_scores.append(next_score)

    print(
        f"Trial {trial + 1}:",
        candidates[next_idx],
        "| CV F1:", round(next_score, 4)
    )

best_position = int(np.argmax(observed_scores))
best_params = candidates[tested_indices[best_position]]
best_cv_score = observed_scores[best_position]

print("\nBest parameters:", best_params)
print("Best cross-validation F1:", round(best_cv_score, 4))
print("Combinations evaluated:", len(tested_indices))

c:\Users\Lenovo\AppData\Local\Python\pythoncore-3.13-64\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\Lenovo\AppData\Local\Python\pythoncore-3.13-64\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\Lenovo\AppData\Local\Python\pythoncore-3.13-64\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\Lenovo\AppData\Local\Python\pythoncore-3.13-64\Lib\site-packages\sklearn\

Trial 6: {'C': 0.1, 'kernel': 'rbf'} | CV F1: 0.6995

Best parameters: {'C': 10, 'kernel': 'rbf'}
Best cross-validation F1: 0.9208
Combinations evaluated: 6


c:\Users\Lenovo\AppData\Local\Python\pythoncore-3.13-64\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\Lenovo\AppData\Local\Python\pythoncore-3.13-64\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\Lenovo\AppData\Local\Python\pythoncore-3.13-64\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\Lenovo\AppData\Local\Python\pythoncore-3.13-64\Lib\site-packages\sklearn\

In [6]:
bayesian_results = pd.DataFrame([
    {
        "Parameters": candidates[idx],
        "Mean CV F1": score
    }
    for idx, score in zip(tested_indices, observed_scores)
])

bayesian_results = bayesian_results.sort_values(
    "Mean CV F1", ascending=False
)

print(bayesian_results.to_string(index=False))

                    Parameters  Mean CV F1
    {'C': 10, 'kernel': 'rbf'}    0.920766
     {'C': 1, 'kernel': 'rbf'}    0.894620
 {'C': 10, 'kernel': 'linear'}    0.867375
  {'C': 1, 'kernel': 'linear'}    0.867375
{'C': 0.1, 'kernel': 'linear'}    0.781119
   {'C': 0.1, 'kernel': 'rbf'}    0.699517


In [7]:
best_model = make_model(best_model_name, best_params)
best_model.fit(X_train, y_train)

test_pred = best_model.predict(X_test)

tp = np.sum((y_test == 1) & (test_pred == 1))
fp = np.sum((y_test == 0) & (test_pred == 1))
fn = np.sum((y_test == 1) & (test_pred == 0))
tn = np.sum((y_test == 0) & (test_pred == 0))

precision = tp / max(tp + fp, 1)
recall = tp / max(tp + fn, 1)
f1 = calculate_f1(y_test, test_pred)
accuracy = np.mean(y_test == test_pred)

print("Optimized model:", best_model_name)
print("Best parameters:", best_params)
print("Test Accuracy:", round(accuracy, 4))
print("Test Precision:", round(precision, 4))
print("Test Recall:", round(recall, 4))
print("Test F1 Score:", round(f1, 4))

Optimized model: SVM
Best parameters: {'C': 10, 'kernel': 'rbf'}
Test Accuracy: 0.6875
Test Precision: 0.7692
Test Recall: 0.8333
Test F1 Score: 0.8


c:\Users\Lenovo\AppData\Local\Python\pythoncore-3.13-64\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
